# ComfyUI + FLUX.3 [klein] (or MiniMax-H3-Image, or Qwen-Image-2.1-Turbo) server for the picture trial

Pushed by `novel ktrial up`. Installs a pinned ComfyUI on one T4, downloads the INT8 FLUX.3 [klein] files (no prompt
upsampler), probes one text-only and one 2-reference picture, then serves ComfyUI through a tunnel behind basic auth
and prints `KTRIAL URL <url>`. It exits on a stop file uploaded by the local side, after the idle limit, or at the
hours cap: cheap checks come first, so a failure costs minutes, not a session.

`session.json` may carry `"model": "minimax-h3"` (`novel ktrial up --model minimax-h3`): then it also installs
ComfyUI-GGUF at a pinned commit, downloads the three GGUF/VAE files instead, and tries the launch layouts of
`kcomfy.MM_LAYOUTS` in order (both cards, fp16; both cards, fp32; one card) until a probe picture is not blank.

`"model": "qwen-image-turbo"` (`novel ktrial up --model qwen-image-turbo`): downloads the INT8 Qwen-Image-2.1-Turbo
diffusion model, the INT8 Qwen3-VL 8B text encoder and the VAE (17.3 GB, no custom node) and tries the launch layouts of
`kcomfy.QW_LAYOUTS` in order (one card, default; one card, fp32) until a probe picture is not blank.

`session.json` may also name `"ngrok_dataset"`: the private dataset that holds the ngrok token (default `ngrok-token`),
for a second ngrok account.

In [ ]:
# Cheap checks first: GPU present, every input dataset present. Nothing is installed yet.
import json, os, shutil, subprocess, sys, time
from pathlib import Path

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
GPUS = [l for l in subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines()
        if l.startswith("GPU ")]
assert GPUS, "no GPU: the kernel must run with a GPU accelerator (NvidiaTeslaT4)"
print("gpus", len(GPUS))

def _find_dataset(slug: str) -> Path:
    candidates = [Path('/kaggle/input') / slug,
                  *Path('/kaggle/input/datasets').glob(f'*/{slug}')]
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError(f"dataset '{slug}' not found under /kaggle/input: attach it to the kernel")

def _secret(slug: str, name: str) -> str:
    p = _find_dataset(slug) / name
    if not p.exists():
        raise FileNotFoundError(f"dataset '{slug}' has no file {name}")
    v = p.read_text().strip()
    assert v, f"{slug}/{name} is empty"
    return v

SRC = _find_dataset('@DATASET@')
WORK = Path('/tmp/ktrial')          # not /kaggle/working: everything there becomes kernel output
WORK.mkdir(parents=True, exist_ok=True)
shutil.copy(SRC / 'kcomfy.py', WORK / 'kcomfy.py')
sys.path.insert(0, str(WORK))
import kcomfy

T_START = time.time()
session = json.loads((SRC / 'session.json').read_text(encoding='utf-8'))
MODEL = session.get("model", kcomfy.FLUX)
assert MODEL in kcomfy.MODELS, f"unknown model {MODEL!r}"
MINIMAX = MODEL == kcomfy.MINIMAX
for k in ("user", "password"):
    assert session.get(k), f"session.json has no {k}"
TUNNEL = session.get("tunnel", "ngrok")
assert TUNNEL in ("ngrok", "cloudflared"), f"unknown tunnel {TUNNEL!r}"
HF_TOKEN = _secret('hf-token', 'HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
NGROK_TOKEN = (_secret(session.get("ngrok_dataset", "ngrok-token"), 'NGROK_AUTHTOKEN')
               if TUNNEL == "ngrok" else None)
# A named Cloudflare tunnel (a fixed https://<cf_hostname>): ngrok's fallback when the local side set images.cf_hostname
CF_HOST = session.get("cf_hostname")
CF_TOKEN = _secret('cloudflare-tunnel', 'CLOUDFLARE_TUNNEL_TOKEN') if CF_HOST else None
if MODEL != kcomfy.FLUX:
    free_gb = shutil.disk_usage('/tmp').free / 1e9
    print(f"disk free on /tmp: {free_gb:.0f} GB, the files need {kcomfy.download_gb(MODEL)} GB")
    assert free_gb > kcomfy.download_gb(MODEL) + 4, f"/tmp has {free_gb:.0f} GB free: too little for the model files"
    import psutil
    print(f"system RAM {psutil.virtual_memory().total / 1e9:.0f} GB, GPUs: {GPUS}")
print("model", MODEL)
print("dataset", SRC, sorted(p.name for p in SRC.iterdir()))
print("session", {k: v for k, v in session.items() if k != "password"}, "tunnel", TUNNEL)

In [ ]:
# Pinned ComfyUI in /tmp. Its requirements leave Kaggle's torch alone ("torch" is unpinned).
COMFY = Path('/tmp/ComfyUI')
t0 = time.time()
if not (COMFY / 'main.py').exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", kcomfy.COMFY_TAG,
                    "https://github.com/comfyanonymous/ComfyUI", str(COMFY)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFY / "requirements.txt")], check=True)
extra = ["huggingface_hub"] + (["pyngrok"] if TUNNEL == "ngrok" else [])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra], check=True)
if MINIMAX:
    CN = COMFY / 'custom_nodes'
    for name, url, commit in kcomfy.MM_NODES:
        d = CN / name
        if not d.exists():
            subprocess.run(["git", "clone", "-q", url, str(d)], check=True)
        subprocess.run(["git", "-C", str(d), "checkout", "-q", commit], check=True)
        head = subprocess.run(["git", "-C", str(d), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
        assert head == commit, f"{name} is at {head}, not {commit}"
        req = d / 'requirements.txt'
        if req.exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)
        print("custom node", name, head)
    (CN / 'ktrial_nodes.py').write_text(kcomfy.KTRIAL_NODES_PY, encoding='utf-8')
print("installed in %.1f min" % ((time.time() - t0) / 60))
print((COMFY / "comfyui_version.py").read_text().strip().splitlines()[-1])

# The T4 is sm_75: a torch without it fails here, in seconds, before the 11 GB download.
CHECK = ("import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda); "
         "print('arch', torch.cuda.get_arch_list()); "
         "print('bf16 supported (maybe emulated)', torch.cuda.is_bf16_supported()); "
         "assert 'sm_75' in torch.cuda.get_arch_list(), 'torch build lacks sm_75'")
subprocess.run([sys.executable, "-c", CHECK], check=True)

In [ ]:
# The model files of the chosen model (FLUX: INT8 diffusion model, text encoder, VAE; the 9B prompt upsampler is NOT
# downloaded. MiniMax: GGUF diffusion model, GGUF text encoder, VAE. Qwen-Image-2.1-Turbo: INT8 diffusion model, INT8
# text encoder, VAE). A file is checked by size after the download.
from huggingface_hub import hf_hub_download

MODELS = COMFY / 'models'
t0 = time.time()
for repo, rel, folder, size in kcomfy.download_plan(MODEL):
    dest = MODELS / folder / Path(rel).name
    if dest.exists() and dest.stat().st_size == size:
        print(f"have {dest.relative_to(MODELS)} {size / 1e9:.2f} GB")
        continue
    t1 = time.time()
    try:
        got_path = Path(hf_hub_download(repo, rel, local_dir=str(MODELS if str(Path(rel).parent) == folder else '/tmp/hfdl'),
                                        token=HF_TOKEN))
    except Exception as e:
        name = type(e).__name__
        if "Gated" in name or "401" in str(e) or "403" in str(e):
            raise RuntimeError(f"{repo} is gated: accept its license on its Hugging Face page with the account behind "
                               f"the hf-token dataset ({name})") from None
        raise
    if got_path != dest:
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(got_path), str(dest))
    got = dest.stat().st_size
    print(f"got  {dest.relative_to(MODELS)} {got / 1e9:.2f} GB in {time.time() - t1:.0f} s")
    assert got == size, f"{rel}: {got} bytes, expected {size}"
print("downloads %.1f min" % ((time.time() - t0) / 60))
print(subprocess.run(["df", "-h", "/tmp"], capture_output=True, text=True).stdout)


In [ ]:
# ComfyUI on the cards of a launch layout. FLUX: ONE card (GPU 1 by default when there are two; GPU 0 is the one
# text-to-speech uses). MiniMax: the layouts of kcomfy.MM_LAYOUTS (both cards first), see the header of that list.
# Qwen-Image-2.1-Turbo: the layouts of kcomfy.QW_LAYOUTS, all on that one card.
GPU = str(session.get("gpu", 1)) if len(GPUS) > 1 else "0"
LAYOUTS = kcomfy.layouts(MODEL, session.get("comfy_args"), flux_visible=GPU)
LOG = Path('/tmp/comfy.log')
comfy_proc = log_f = None

def log_tail(n=60):
    return "\n".join(LOG.read_text(errors="replace").splitlines()[-n:])

def start_comfy(layout):
    global comfy_proc, log_f
    env = dict(os.environ)
    if layout["visible"]:
        env["CUDA_VISIBLE_DEVICES"] = layout["visible"]
    env.update(layout.get("env") or {})     # e.g. KTRIAL_VAE_DEVICE, read by ktrial_nodes.py in this process only
    log_f = open(LOG, "w")
    comfy_proc = subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188",
                                   "--disable-auto-launch", *layout["args"]],
                                  cwd=str(COMFY), env=env, stdout=log_f, stderr=subprocess.STDOUT)
    print("ComfyUI pid", comfy_proc.pid, "layout", layout["name"], "CUDA_VISIBLE_DEVICES", layout["visible"] or "(both)",
          "args", layout["args"], "env", layout.get("env") or {}, flush=True)
    probe_client = kcomfy.Comfy("http://127.0.0.1:8188", timeout=10, retries=0)
    deadline = time.time() + 600
    while True:
        if comfy_proc.poll() is not None:
            raise RuntimeError(f"ComfyUI exited with {comfy_proc.returncode}:\n{log_tail()}")
        try:
            return probe_client.ping()
        except kcomfy.ComfyError:
            if time.time() > deadline:
                raise RuntimeError(f"ComfyUI did not answer in 10 min:\n{log_tail()}")
            time.sleep(3)

def stop_comfy():
    comfy_proc.terminate()
    try:
        comfy_proc.wait(timeout=60)
    except subprocess.TimeoutExpired:
        comfy_proc.kill()
    log_f.close()


In [ ]:
# Probe pictures, all 1376x768, per layout. FLUX: a text-only picture, then a picture with two references (made
# text-only first); fails fast on a black picture or a reference run slower than MAX_SECONDS. MiniMax: a text-only
# picture, then one with the first as its reference (`<Picture 1>`); a blank picture, an error or a timeout moves on to
# the next layout (a fresh ComfyUI; the downloads stay).
from PIL import Image, ImageStat
import io

MAX_SECONDS = 180
MIN_STD = 8.0
W, H = 1376, 768
PROBE = Path('/kaggle/working/probe')
PROBE.mkdir(parents=True, exist_ok=True)
PICTURE_TIMEOUT = kcomfy.probe_timeout(MODEL)

def looks_blank(data: bytes) -> bool:
    img = Image.open(io.BytesIO(data)).convert("L")
    return max(ImageStat.Stat(img).stddev) < MIN_STD

def probe(name, prompt, seed, refs=()):
    data, took = local.generate(kcomfy.adapt_prompt(MODEL, prompt), seed, W, H, list(refs), f"probe/{name}",
                                timeout=PICTURE_TIMEOUT, model=MODEL)
    (PROBE / f"{name}.png").write_bytes(data)
    print(f"probe {name}: {took:.1f}s", flush=True)
    assert not looks_blank(data), f"probe {name}: the picture is blank (std below {MIN_STD}); see {PROBE}"
    return data, took

def vram_line():
    out = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip().replace("\n", " | ")
    ram = ""
    try:
        import psutil
        vm = psutil.virtual_memory()
        ram = f" ram used {vm.used / 1e9:.1f} of {vm.total / 1e9:.1f} GB"
    except ImportError:
        pass
    return out + ram

STYLE = "Detailed digital painting, soft natural light."

def run_probes():
    if MODEL != kcomfy.FLUX:         # MiniMax and Qwen: one text-only picture, then one with a reference
        text, _ = probe("text", f"{STYLE} A lighthouse on a rocky coast at dusk, waves breaking below, gulls in the sky.", 1)
        name = local.upload_bytes(text, "probe_ref_text.png")
        probe("one_ref", f"{STYLE} The lighthouse of {kcomfy.ref_token(0)} on a stormy night, heavy rain, a ship far out.",
              2, [name])
        return
    probe("text", f"{STYLE} A lighthouse on a rocky coast at dusk, waves breaking below, gulls in the sky.", 1)
    person, _ = probe("ref_person", f"{STYLE} Head and shoulders portrait of a young man with short black hair, "
                      "a dark green wool coat and a grey scarf, plain light background.", 2)
    room, _ = probe("ref_room", f"{STYLE} An empty old library with tall wooden shelves, a round table, "
                    "warm lamp light, no people.", 3)
    ref_names = [local.upload_bytes(person, "probe_ref_person.png"), local.upload_bytes(room, "probe_ref_room.png")]
    _, took = probe("two_refs", f"{STYLE} The man from {kcomfy.ref_token(0)} sits at the round table in the library "
                    f"from {kcomfy.ref_token(1)}, reading a book, medium shot.", 4, ref_names)
    assert took < MAX_SECONDS, f"reference picture took {took:.0f}s, limit {MAX_SECONDS}s: too slow for the trial"

chosen, failures = None, []
for layout in LAYOUTS:
    t_layout = time.time()
    try:
        stats = start_comfy(layout)
        print(json.dumps(stats, indent=1)[:1500])
        local = kcomfy.Comfy("http://127.0.0.1:8188", timeout=60, retries=2)
        run_probes()
        chosen = layout
        break
    except Exception as e:
        failures.append((layout["name"], f"{type(e).__name__}: {str(e)[:300]}"))
        print(f"KTRIAL layout {layout['name']} failed after {(time.time() - t_layout) / 60:.1f} min: "
              f"{type(e).__name__}: {str(e)[:600]}", flush=True)
        print(log_tail(80), flush=True)
        if len(LAYOUTS) == 1:
            raise
        try:
            print("vram:", vram_line())
        except Exception:
            pass
        stop_comfy()
if chosen is None:
    if MODEL == kcomfy.FLUX:
        raise RuntimeError("no launch layout worked: " + json.dumps(failures))
    # MiniMax and Qwen: keep a server up anyway (the first layout, no probe) so the graph and settings can be tried live through
    # the tunnel instead of by restarting the kernel; the idle limit below still ends it.
    print("KTRIAL DEGRADED: no launch layout passed its probes: " + json.dumps(failures), flush=True)
    chosen = LAYOUTS[0]
    print(json.dumps(start_comfy(chosen), indent=1)[:300])
    local = kcomfy.Comfy("http://127.0.0.1:8188", timeout=60, retries=2)
print("KTRIAL LAYOUT", chosen["name"], "args", chosen["args"], "visible", chosen["visible"] or "(both)", flush=True)
print("KTRIAL probes ok after %.1f min since the notebook started" % ((time.time() - T_START) / 60), flush=True)
print(vram_line(), flush=True)


# The second T4 (Qwen only): a second ComfyUI on the other card, port 8190, reached through the proxy at /g2, so the
# local side can draw two pictures at once. It loads its own copy of the model, so it starts only when enough system
# RAM is free and its probe passes; otherwise the server is the one card it always was.
second_proc = log2 = None
SECOND_PORT = 8190
SECOND_RAM_GB = 14
if MODEL == kcomfy.QWEN and len(GPUS) > 1 and session.get("second_gpu", True):
    try:
        import psutil
        free_gb = psutil.virtual_memory().available / 1e9
        if free_gb < SECOND_RAM_GB:
            raise RuntimeError(f"{free_gb:.0f} GB RAM free, {SECOND_RAM_GB} needed")
        other = "1" if (chosen["visible"] or GPU) == "0" else "0"
        env2 = dict(os.environ, CUDA_VISIBLE_DEVICES=other)
        env2.update(chosen.get("env") or {})
        log2 = open('/tmp/comfy2.log', 'w')
        second_proc = subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", str(SECOND_PORT),
                                        "--disable-auto-launch", *chosen["args"]],
                                       cwd=str(COMFY), env=env2, stdout=log2, stderr=subprocess.STDOUT)
        ping2 = kcomfy.Comfy(f"http://127.0.0.1:{SECOND_PORT}", timeout=10, retries=0)
        t_end = time.time() + 600
        while True:
            if second_proc.poll() is not None:
                raise RuntimeError(f"ComfyUI exited with {second_proc.returncode}")
            try:
                ping2.ping()
                break
            except kcomfy.ComfyError:
                if time.time() > t_end:
                    raise RuntimeError("no answer in 10 min")
                time.sleep(3)
        local2 = kcomfy.Comfy(f"http://127.0.0.1:{SECOND_PORT}", timeout=60, retries=2)
        data2, took2 = local2.generate(kcomfy.adapt_prompt(MODEL, f"{STYLE} A lighthouse on a rocky coast at dusk."), 3,
                                       W, H, [], "probe/second", timeout=PICTURE_TIMEOUT, model=MODEL)
        assert not looks_blank(data2), "the second card's probe picture is blank"
        print(f"KTRIAL SECOND CARD on (GPU {other}): probe {took2:.0f}s; {vram_line()}", flush=True)
    except Exception as e:
        print(f"KTRIAL SECOND CARD off: {type(e).__name__}: {str(e)[:300]}", flush=True)
        if second_proc is not None and second_proc.poll() is None:
            second_proc.terminate()
        second_proc = None

In [ ]:
# Basic auth is enforced by kcomfy's reverse proxy on :8189, so it does not depend on the tunnel's plan
# (ngrok's basic-auth Traffic Policy action is not included in its free plan). The tunnel points at the proxy.
import re, threading, urllib.request

proxy = kcomfy.serve_auth_proxy(8189, "http://127.0.0.1:8188", session["user"], session["password"],
                                routes={kcomfy.SECOND_PREFIX: f"http://127.0.0.1:{SECOND_PORT}"} if second_proc else None)
tunnel_proc = None


def cloudflared_bin():
    CF = Path('/tmp/cloudflared')
    if not CF.exists():
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/download/2026.11.0/cloudflared-linux-amd64", CF)
        CF.chmod(0o755)
    return CF


def open_named_tunnel():
    """The named tunnel (its ingress, set in the Cloudflare dashboard, points https://CF_HOST at localhost:8189)."""
    proc = subprocess.Popen([str(cloudflared_bin()), "tunnel", "--no-autoupdate", "run", "--token", CF_TOKEN],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    threading.Thread(target=lambda: [None for _ in proc.stdout], daemon=True).start()
    url = "https://" + CF_HOST
    kcomfy.Comfy(url, session["user"], session["password"], timeout=30, retries=8).ping()
    return url, proc


def open_cloudflared():
    """A Cloudflare quick tunnel: no account, a random https://....trycloudflare.com address."""
    CF = cloudflared_bin()
    tunnel_proc = subprocess.Popen([str(CF), "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8189"],
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    PUBLIC, t_end = None, time.time() + 120
    while PUBLIC is None and time.time() < t_end:
        line = tunnel_proc.stdout.readline()
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line or "")
        if m:
            PUBLIC = m.group(0)
    assert PUBLIC, "cloudflared printed no trycloudflare.com URL in 2 min"
    # keep draining its output so the pipe never fills
    threading.Thread(target=lambda: [None for _ in tunnel_proc.stdout], daemon=True).start()
    return PUBLIC, tunnel_proc


def open_ngrok():
    """ngrok on the account's fixed dev domain (the address the local side knows), else a random one."""
    ngrok.set_auth_token(NGROK_TOKEN)
    domain = session.get("domain")
    if domain:
        url = ngrok.connect(addr="8189", url="https://" + domain).public_url.replace("http://", "https://", 1)
        assert url == "https://" + domain, f"ngrok opened {url}, not https://{domain}"
    else:
        url = ngrok.connect("8189", "http").public_url.replace("http://", "https://", 1)
    kcomfy.Comfy(url, session["user"], session["password"], timeout=30, retries=6).ping()
    return url


# The order: the named Cloudflare tunnel (fixed https://CF_HOST, the primary), then ngrok (fixed dev domain), then a
# Cloudflare quick tunnel (random address: the local side reads it from this log, `novel ktrial url`). The local side
# tries the first two addresses by itself.
if TUNNEL == "ngrok":
    from pyngrok import ngrok
ORDER = (["named"] if CF_HOST else []) + (["ngrok"] if TUNNEL == "ngrok" else []) + ["quick"]
PUBLIC = None
for way in ORDER:
    try:
        if way == "named":
            PUBLIC, tunnel_proc = open_named_tunnel()
        elif way == "ngrok":
            PUBLIC = open_ngrok()
        else:
            PUBLIC, tunnel_proc = open_cloudflared()
        print(f"KTRIAL TUNNEL {way} {PUBLIC}", flush=True)
        if way == "quick" and len(ORDER) > 1:
            print("KTRIAL FALLBACK cloudflared " + PUBLIC, flush=True)
        break
    except Exception as e:
        print(f"KTRIAL tunnel {way} failed: {type(e).__name__}: {str(e)[:300]}", flush=True)
        if way == "ngrok":
            try:
                ngrok.kill()
            except Exception:
                pass
        elif tunnel_proc is not None:
            tunnel_proc.terminate()
            tunnel_proc = None
        if way == "quick":
            raise
TUNNEL = "ngrok" if way == "ngrok" else "cloudflared"      # what the stop code below closes

# Check the whole path once, from outside in: tunnel -> proxy (auth) -> ComfyUI.
remote = kcomfy.Comfy(PUBLIC, session["user"], session["password"], timeout=30, retries=6)
remote.ping()
try:
    kcomfy.Comfy(PUBLIC, timeout=30, retries=2).ping()
    raise AssertionError("the tunnel answered without a password")
except kcomfy.ComfyError as e:
    assert "401" in str(e), e
print("KTRIAL URL " + PUBLIC, flush=True)

In [ ]:
# Keep alive until the stop file, the idle limit or the hours cap. Then stop everything and keep the log.
STOP = COMFY / 'input' / kcomfy.STOP_NAME
IDLE_MIN = float(session.get("idle_minutes", 20))
MAX_H = float(session.get("max_hours", 6))
started = last_active = time.time()
last_beat, seen_hist, reason = 0.0, None, None
try:
    while reason is None:
        now = time.time()
        try:
            busy = local.queue_size()
            hist = local.history_count()
        except kcomfy.ComfyError as e:
            print("KTRIAL ComfyUI not answering:", e, flush=True)
            if comfy_proc.poll() is not None:
                reason = f"ComfyUI exited with {comfy_proc.returncode}"
                break
            busy, hist = 0, seen_hist
        if busy or hist != seen_hist:
            last_active = now
        seen_hist = hist
        last_active = max(last_active, proxy.last_request)
        idle = (now - last_active) / 60
        if STOP.exists():
            reason = "stop file"
        elif idle >= IDLE_MIN:
            reason = f"idle {idle:.0f} min"
        elif now - started >= MAX_H * 3600:
            reason = f"max {MAX_H:g} h"
        elif now - last_beat >= 300:
            print(f"KTRIAL alive, idle {idle:.0f} min, done {hist}", flush=True)
            last_beat = now
        if reason is None:
            time.sleep(30)
finally:
    print("KTRIAL stopping:", reason, flush=True)
    try:
        if TUNNEL == "ngrok":
            ngrok.kill()
        elif tunnel_proc is not None:
            tunnel_proc.terminate()
    except Exception as e:
        print("tunnel stop:", e)
    proxy.shutdown()
    if second_proc is not None:
        second_proc.terminate()
    comfy_proc.terminate()
    try:
        comfy_proc.wait(timeout=60)
    except subprocess.TimeoutExpired:
        comfy_proc.kill()
    log_f.close()
    shutil.copy(LOG, '/kaggle/working/comfy.log')
    print("KTRIAL stopped", flush=True)